In [27]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [28]:
!pip install faiss-cpu sentence-transformers

In [29]:
import os
import sys
from pathlib import Path

PROJECT_PATH = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

if str(PROJECT_PATH) not in sys.path:
    sys.path.insert(0, str(PROJECT_PATH))

print("Project path:", PROJECT_PATH)

Project path: /content/drive/MyDrive/rag-chatbot-evaluation-framework


In [30]:
import pandas as pd

PROJECT_DIR = Path("/content/drive/MyDrive/rag-chatbot-evaluation-framework")
SRC_PATH = PROJECT_DIR / "src"

print("PROJECT_PATH exists:", PROJECT_DIR.exists())
print("SRC_PATH exists:", SRC_PATH.exists())
print("Files in src:", os.listdir(SRC_PATH))

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from document_loader import load_markdown_documents, validate_documents
from text_splitter import create_chunks, get_chunk_statistics
from vector_store import FaissVectorStore
from rag_pipeline import SimpleRAGPipeline
from evaluator import add_pass_fail_flags, evaluate_dataset, summarize_results


DOCUMENT_DIR = PROJECT_DIR / "data" / "documents"
EVALUATION_FILE = PROJECT_DIR / "data" / "evaluation" / "evaluation_questions.csv"
RESULTS_DIR = PROJECT_DIR / "results"


docs = load_markdown_documents(DOCUMENT_DIR)
validate_documents(docs)

evaluation_df = pd.read_csv(EVALUATION_FILE)
evaluation_df.head()

PROJECT_PATH exists: True
SRC_PATH exists: True
Files in src: ['__init__.py', 'rag_pipeline.py', 'text_splitter.py', 'document_loader.py', 'vector_store.py', '__pycache__', 'evaluator.py', 'report_generator.py']
Document validation passed. Loaded 5 documents.


,question,expected_answer,expected_source,expected_keywords,question_type
0,What is the return period?,Customers can return most products within 30 d...,return_policy.md,30 days;return;delivery,normal
1,What condition must returned items be in?,"Returned items must be unused, undamaged, and ...",return_policy.md,unused;undamaged;original packaging,normal
2,How long does it usually take to process a ref...,Refunds are usually processed within 5 to 10 b...,return_policy.md,5 to 10 business days;refund;received,normal
3,Can customized products be returned?,Customized products cannot be returned.,return_policy.md,customized products;cannot be returned,normal
4,What should a customer do if a product arrives...,Customers should contact customer support with...,return_policy.md,7 days;customer support;photos,normal


In [31]:
from typing import overload
experiment_rows = []
for chunk_size in [300, 500, 800]:
  for top_k in [1, 3, 5]:
    chunks = create_chunks(docs, chunk_size=chunk_size, overlap=100)
    chunk_statc = get_chunk_statistics(chunks)

    vector_store = FaissVectorStore()
    vector_store.build_index(chunks)

    rag = SimpleRAGPipeline(vector_store, top_k=top_k)

    evaluation_results = evaluate_dataset(rag, evaluation_df)
    evaluation_results = add_pass_fail_flags(evaluation_results)
    summary = summarize_results(evaluation_results).iloc[0].to_dict()

    experiment_rows.append({
        'chunk_size': chunk_size,
        'top_k': top_k,
        'total_chunks': len(chunks),
        'avg_source_match': summary['avg_source_match'],
        'avg_keyword_recall': summary['avg_keyword_recall'],
        'avg_unanswerable_safe': summary['avg_unanswerable_safe'],
        'overall_pass_rate': summary['overall_pass_rate']
    })

experiment_results = pd.DataFrame(experiment_rows)
experiment_results.sort_values('overall_pass_rate', ascending=False)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

,chunk_size,top_k,total_chunks,avg_source_match,avg_keyword_recall,avg_unanswerable_safe,overall_pass_rate
1,300,3,15,1.000000,0.875000,0.0,0.87500
8,800,5,5,1.000000,0.875000,0.0,0.87500
2,300,5,15,1.000000,0.875000,0.0,0.87500
4,500,3,10,1.000000,0.875000,0.0,0.87500
5,500,5,10,1.000000,0.875000,0.0,0.87500
7,800,3,5,1.000000,0.875000,0.0,0.87500
0,300,1,15,1.000000,0.786458,0.0,0.84375
3,500,1,10,0.965517,0.828125,0.0,0.84375
6,800,1,5,0.931034,0.812500,0.0,0.81250


In [32]:
!pip install pytest
%cd /content/drive/MyDrive/rag-chatbot-evaluation-framework
!ls
!pytest tests/

/content/drive/MyDrive/rag-chatbot-evaluation-framework
app.py	notebooks	 README.md  requirements.txt  src    venv
data	PHASE1_SETUP.md  reports    results	      tests
============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0
rootdir: /content/drive/MyDrive/rag-chatbot-evaluation-framework
plugins: anyio-4.14.0, typeguard-4.5.2, langsmith-0.9.1
collected 17 items                                                             

tests/test_document_loader.py ...                                        [ 17%]
tests/test_evaluator.py .......                                          [ 58%]
tests/test_rag_pipeline.py ..                                            [ 70%]
tests/test_retrieval.py .                                                [ 76%]
tests/test_text_splitter.py ....                                         [100%]

============================= 17 passed in 18.73s ==============================


In [33]:
!pytest tests/ -v

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/drive/MyDrive/rag-chatbot-evaluation-framework
plugins: anyio-4.14.0, typeguard-4.5.2, langsmith-0.9.1
collected 17 items                                                             

tests/test_document_loader.py::test_load_markdown_documents PASSED       [  5%]
tests/test_document_loader.py::test_validate_documents PASSED            [ 11%]
tests/test_document_loader.py::test_get_document_summary PASSED          [ 17%]
tests/test_evaluator.py::test_keyword_recall_full_match PASSED           [ 23%]
tests/test_evaluator.py::test_keyword_recall_partial_match PASSED        [ 29%]
tests/test_evaluator.py::test_source_match_success PASSED                [ 35%]
tests/test_evaluator.py::test_source_match_failure PASSED                [ 41%]
tests/test_evaluator.py::test_unanswerable_safe_answ

In [34]:
!pip install streamlit
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼
changed 22 packages in 2s
⠼
⠼3 packages are looking for funding
⠼  run `npm fund` for details
⠼

In [26]:
!pkill -f streamlit

!streamlit run app.py --server.port 8501 --server.address 0.0.0.0 &

!sleep 5

print("LocalTunnel password / Colab IP:")
!curl ipv4.icanhazip.com

print("Open the URL below:")
!lt --port 8501

/content/drive/MyDrive/rag-chatbot-evaluation-framework


2026-07-06 08:29:08.100 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.106.57.186:8501

  Stopping...


In [35]:
!pkill -f streamlit

In [ ]:
!curl ipv4.icanhazip.com
!lt --port 8501

34.106.57.186
your url is: https://nice-grapes-enjoy.loca.lt
